## 00 — The Viewer

This is the assembly notebook. Every component we built across Modules 00–05 is brought together here into one clean, readable file.

This corrected version uses only the existing `data/ne_10m_railroads.geojson` file. It builds the LOD levels in memory instead of reading or creating separate `data/lod/railroads_*.geojson` files.

No new concepts. The goal is:
- See all the pieces side by side
- Produce a working viewer with styled output
- Understand the full data flow in one place

Read through the code before running it. Every line should be recognizable.


## 1. Imports and Paths

In [1]:
import json
import math
import time
import copy
from pathlib import Path
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets


def find_raw_path():
    """Find data/ne_10m_railroads.geojson from the current notebook folder or parents."""
    cwd = Path.cwd().resolve()

    for base in [cwd] + list(cwd.parents):
        candidate = base / "data" / "ne_10m_railroads.geojson"
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/ne_10m_railroads.geojson. "
        "Keep the notebook inside the project folder that contains the data folder."
    )


RAW_PATH = find_raw_path()

# These LODs are built in memory from the one raw GeoJSON file.
# No data/lod folder and no extra railroad_*.geojson files are required.
LOD_CONFIG = [
    {"name": "coarse",     "zoom_max": 3,  "epsilon": 1.0,  "max_scalerank": 4},
    {"name": "medium",     "zoom_max": 6,  "epsilon": 0.5,  "max_scalerank": None},
    {"name": "fine",       "zoom_max": 10, "epsilon": 0.15, "max_scalerank": None},
    {"name": "extra_fine", "zoom_max": 99, "epsilon": 0.03, "max_scalerank": None},
]

print("Using raw railroad file:", RAW_PATH)


Using raw railroad file: /workspaces/ricardoayala2510-Spatial-Data-Mapping/assigments completed/03-Data_Manager/data/ne_10m_railroads.geojson


## 2. Geometry and Index Utilities

In [2]:
def iter_points(coords):
    """Yield [lon, lat] points from LineString or MultiLineString coordinates."""
    if not coords:
        return

    if isinstance(coords[0], (int, float)):
        yield coords
    else:
        for part in coords:
            yield from iter_points(part)


def feature_bbox(feature):
    points = list(iter_points(feature["geometry"]["coordinates"]))

    if not points:
        return [0, 0, 0, 0]

    lons = [p[0] for p in points]
    lats = [p[1] for p in points]
    return [min(lons), min(lats), max(lons), max(lats)]


def leaflet_bounds_to_bbox(bounds):
    """Convert ipyleaflet [[lat_min,lon_min],[lat_max,lon_max]] to [lon_min,lat_min,lon_max,lat_max]."""
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]


class GridIndex:
    """Uniform grid spatial index. Assigns features to 10° cells for fast viewport queries."""

    CELL_SIZE = 10.0

    def __init__(self):
        self.cells = {}

    def _cells(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        cs = self.CELL_SIZE
        col_min, col_max = int((lon_min + 180) / cs), int((lon_max + 180) / cs)
        row_min, row_max = int((lat_min +  90) / cs), int((lat_max +  90) / cs)
        return [(c, r) for c in range(col_min, col_max + 1) for r in range(row_min, row_max + 1)]

    def build(self, features):
        self.cells = {}
        for idx, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)):
                self.cells.setdefault(cell, []).append((idx, f))

    def query(self, viewport_bbox):
        seen, results = set(), []
        for cell in self._cells(viewport_bbox):
            for idx, f in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(f)
        return results


## 3. Load Data and Build Indexes

In [3]:
def point_distance_to_segment(p, a, b):
    """Distance from point p to line segment ab in lon/lat degree space."""
    px, py = p
    ax, ay = a
    bx, by = b

    dx = bx - ax
    dy = by - ay

    if dx == 0 and dy == 0:
        return math.hypot(px - ax, py - ay)

    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0, min(1, t))

    proj_x = ax + t * dx
    proj_y = ay + t * dy
    return math.hypot(px - proj_x, py - proj_y)


def douglas_peucker(points, epsilon):
    """Simplify one LineString with the Douglas-Peucker algorithm."""
    if len(points) <= 2:
        return points

    start = points[0]
    end = points[-1]

    max_dist = -1
    index = 0

    for i in range(1, len(points) - 1):
        dist = point_distance_to_segment(points[i], start, end)
        if dist > max_dist:
            max_dist = dist
            index = i

    if max_dist > epsilon:
        left = douglas_peucker(points[:index + 1], epsilon)
        right = douglas_peucker(points[index:], epsilon)
        return left[:-1] + right

    return [start, end]


def simplify_geometry(geometry, epsilon):
    """Simplify LineString and MultiLineString geometries without writing any files."""
    if not geometry:
        return geometry

    geom_type = geometry.get("type")
    coords = geometry.get("coordinates")

    if geom_type == "LineString":
        if len(coords) < 2:
            return geometry
        return {
            "type": "LineString",
            "coordinates": douglas_peucker(coords, epsilon),
        }

    if geom_type == "MultiLineString":
        simplified_parts = []
        for part in coords:
            if len(part) >= 2:
                simplified_parts.append(douglas_peucker(part, epsilon))
        return {
            "type": "MultiLineString",
            "coordinates": simplified_parts,
        }

    return geometry


def get_scalerank(feature):
    try:
        return int(feature.get("properties", {}).get("scalerank", 99))
    except Exception:
        return 99


# Load the one raw file only.
with open(RAW_PATH) as f:
    raw_data = json.load(f)

raw_features = raw_data["features"]

lod_features = {}
lod_indexes = {}

for cfg in LOD_CONFIG:
    t0 = time.perf_counter()

    features = []
    for feature in raw_features:
        if cfg["max_scalerank"] is not None and get_scalerank(feature) > cfg["max_scalerank"]:
            continue

        new_feature = copy.deepcopy(feature)
        new_feature["geometry"] = simplify_geometry(new_feature["geometry"], cfg["epsilon"])
        features.append(new_feature)

    idx = GridIndex()
    idx.build(features)

    elapsed = time.perf_counter() - t0
    lod_features[cfg["name"]] = features
    lod_indexes[cfg["name"]] = idx

    total_points = sum(1 for f in features for _ in iter_points(f["geometry"]["coordinates"]))
    print(f"  {cfg['name']:<12}  {len(features):>6,} features  {total_points:>10,} pts  {elapsed:.2f}s")

print("\nReady. Built all LOD levels in memory from the single raw GeoJSON file.")


  coarse         2,845 features       5,690 pts  1.51s
  medium        25,413 features      50,969 pts  5.34s
  fine          25,413 features      53,182 pts  3.87s
  extra_fine    25,413 features      75,577 pts  5.78s

Ready. Built all LOD levels in memory from the single raw GeoJSON file.


## 4. LOD Selection

In [4]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    for cfg in LOD_CONFIG:
        if z <= cfg["zoom_max"]:
            return cfg["name"]
    return LOD_CONFIG[-1]["name"]

## 5. Styling

We use `scalerank` to vary line weight — more important lines render thicker, just like a real map.

In [5]:
def style_callback(feature):
    rank = feature["properties"].get("scalerank", 5)
    weight = max(0.5, 2.5 - rank * 0.3)   # rank 1 → thick, rank 7 → thin
    return {
        "color":   "#333333",
        "weight":  weight,
        "opacity": 0.85,
    }

## 6. The Map

In [6]:
current_lod = get_lod(5)

m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style_callback=style_callback
)
m.add(layer)

# Status bar
lbl_lod      = widgets.Label()
lbl_zoom     = widgets.Label()
lbl_features = widgets.Label()
lbl_time     = widgets.Label()
status       = widgets.HBox([lbl_lod, lbl_zoom, lbl_features, lbl_time])

def update(*args):
    global current_lod
    if not m.bounds:
        return

    current_lod    = get_lod(m.zoom)
    vp             = leaflet_bounds_to_bbox(m.bounds)

    t0             = time.perf_counter()
    visible        = lod_indexes[current_lod].query(vp)
    elapsed_ms     = (time.perf_counter() - t0) * 1000

    layer.data          = {"type": "FeatureCollection", "features": visible}
    lbl_lod.value       = f"LOD: {current_lod}"
    lbl_zoom.value      = f"  zoom: {int(math.floor(m.zoom))}"
    lbl_features.value  = f"  features: {len(visible):,}"
    lbl_time.value      = f"  query: {elapsed_ms:.1f}ms"

m.observe(update, names=["zoom", "bounds"])
update()

widgets.VBox([m, status])

## Exercise A

Add a second style dimension: color the lines by `category` property.

1. Find the unique `category` values in the fine LOD features
2. Assign a distinct color to each
3. Update `style_callback` to use category color + scalerank weight together

The result should show the railroad network colored by line type.


In [7]:
# Add category-based coloring to style_callback

# 1) Use the in-memory fine LOD features. No extra railroad_fine.geojson file is needed.
fine_features = lod_features["fine"]

categories = sorted({
    feature.get("properties", {}).get("category", "unknown")
    for feature in fine_features
})

# 2) Assign each category a distinct color.
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf",
]

category_colors = {
    category: palette[i % len(palette)]
    for i, category in enumerate(categories)
}

print("Category colors:")
for category, color in category_colors.items():
    print(f"  {category}: {color}")


# 3) Update style_callback so color comes from category and weight comes from scalerank.
def style_callback(feature):
    props = feature.get("properties", {})

    category = props.get("category", "unknown")
    rank = props.get("scalerank", 5)

    weight = max(0.5, 2.5 - rank * 0.3)   # rank 1 → thick, rank 7 → thin

    return {
        "color": category_colors.get(category, "#333333"),
        "weight": weight,
        "opacity": 0.85,
    }


# Apply the new style function to the existing GeoJSON layer.
layer.style_callback = style_callback

# Force the layer to redraw using the current visible data.
layer.data = {
    "type": "FeatureCollection",
    "features": layer.data.get("features", [])
}

# Refresh the current viewport.
update()


Category colors:
  0: #1f77b4
  1: #ff7f0e
  2: #2ca02c
  3: #d62728
  6: #9467bd
  7: #8c564b
  9: #e377c2


## Exercise B

Add a tooltip: when the user hovers over a railroad feature, show its `category` and `scalerank` in a widget below the map.

Hint: use the GeoJSON layer's `on_hover` event.

In [8]:
# Add hover tooltip showing category and scalerank

hover_label = widgets.Label(value="Hover over a railroad feature to see its category and scalerank.")


def handle_hover(*args, **kwargs):
    """Update the label when the mouse hovers over a GeoJSON feature."""
    feature = kwargs.get("feature")

    # ipyleaflet callback signatures can vary slightly, so also inspect args.
    if feature is None:
        for arg in args:
            if isinstance(arg, dict) and "properties" in arg:
                feature = arg
                break

    if not feature:
        hover_label.value = "Hover over a railroad feature to see its category and scalerank."
        return

    props = feature.get("properties", {})
    category = props.get("category", "unknown")
    scalerank = props.get("scalerank", "unknown")

    hover_label.value = f"category: {category}   |   scalerank: {scalerank}"


# Register the hover callback on the existing layer.
layer.on_hover(handle_hover)

# Show the existing map, status bar, and tooltip together.
widgets.VBox([m, status, hover_label])


## Check Your Understanding

The viewer re-runs `get_lod()` and queries the grid on **every** bounds change — even if the user only pans a few pixels and the LOD hasn't changed.

Describe two optimizations you could apply to skip redundant work:
1. One that avoids calling `get_lod()` unnecessarily
2. One that avoids re-querying the grid when the viewport barely moved

For each, what is the tradeoff?

---

**Answer:**

1. To avoid calling `get_lod()` unnecessarily, separate the event handlers. Only run `get_lod()` when the `zoom` value changes. If the user only pans and the zoom stays the same, keep using the current LOD value. The tradeoff is that the code becomes a little more complex because zoom changes and bounds changes need different logic.

2. To avoid re-querying the grid when the viewport barely moved, cache the last viewport bounding box and only run `index.query()` again if the new bounds changed by more than a chosen threshold, such as a fraction of a degree or one grid cell. The tradeoff is that the map may temporarily show slightly stale results during very small pans, but it avoids doing repeated work that would produce almost the same visible feature set.


## Next

In [01 — What We Built](./01-What_We_Built.ipynb), we step back, measure the system's remaining limitations, and document every decision we made.